# Show the `lsst.analysis.tools` plots produced by DRP stage1 (single-visit)

- author : Sylvie Dagoret-Campagne
- creation date : 2026-11-06 : stage 1
- purpose : display, from the Butler, the plots made by the `lsst.analysis.tools` tasks in `DRP.yaml#stage1-single-visit` (dataset types ending with `Plot`, storage class `Plot` = matplotlib `Figure`), in order to understand what the steps (1c, 1d) do
- next : same notebook for stage 2 (recalibrate)

- the quantum graph for DRP.yaml#stage1-single-visit: https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_40/drp_pipe/LSSTCam/DRP/pipeline_drp_pipe_LSSTCam_DRP_stage1-single-visit.pdf
- all quantum graphs : https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_43/drp_pipe/LSSTCam/DRP/

## Import

In [ ]:
import os
import re
import collections
import logging
import warnings

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from lsst.daf.butler import Butler

In [ ]:
# Turn the Butler.collections FutureWarning into an error (same convention as the CheckCollections notebooks)
warnings.filterwarnings(
    'error',
    category=FutureWarning,
    message='.*Butler.collections.*',
)

# Configure logging for Rubin Science Pipelines
logging.basicConfig(
    level=logging.WARNING,
    format='%(name)s %(levelname)s: %(message)s',
    force=True,
)

In [ ]:
butler_url = 'main'
butler = Butler(butler_url)

## Configuration

In [ ]:
NB_TAG = 'ShowPlots_DRPstage1'

username = os.getenv('USER')

# stage 1 output collection
collection_path = (
    f'u/{username}/LSSTCam/runcosmos01_20261103_tract9813patchsel_RunCosmos9patches/stage1'
)

# Quantum diagrams of the DRP pipeline steps
DIAGRAM_DIR = '../../../../notebooks/fig_diagrams'

# Save the plots retrieved from the Butler (PNG + PDF)
SAVE_FIGS = True
FIGDIR = f'figs_{NB_TAG}'
if SAVE_FIGS:
    os.makedirs(FIGDIR, exist_ok=True)

print('collection :', collection_path)

In [ ]:
def ShowOneImage(filename_img, title=None, caption=None):
    '''Display a PNG file (here a quantum diagram) in the notebook.'''
    if not os.path.exists(filename_img):
        print('File not found :', os.path.abspath(filename_img))
        return
    with open(filename_img, 'rb') as f:
        data = f.read()
    if title is not None:
        display(Markdown(f'### {title}'))
    display(Image(data=data))
    if caption is not None:
        display(Markdown(f'*{caption}*'))

## Quantum diagrams of the stage1 steps

In [ ]:
diagrams = [
    ('step1a-single-visit-detectors', 'DRP step1a/detectors'),
    ('step1b-single-visit-visits',    'DRP step1b/visits'),
    ('step1c-single-visit-tracts',    'DRP step1c/tracts'),
    ('step1d-single-visit-global',    'DRP step1d/global'),
]

for step, label in diagrams:
    filename_diagram = os.path.join(DIAGRAM_DIR, f'pipeline_drp_pipe_LSSTCam_DRP_{step}.png')
    ShowOneImage(filename_diagram, title=label)

## Find the Plot datasets in the stage1 collection

In [ ]:
exists = collection_path in butler.collections.query(collection_path)
print(f'exists = {exists} for {collection_path}')

all_refs = list(
    butler.registry.queryDatasets(
        datasetType='*',
        collections=collection_path,
        findFirst=False,
    )
)
print('Total number of datasets :', len(all_refs))

# Keep only the plots (dataset types ending with 'Plot')
plot_refs = [r for r in all_refs if r.datasetType.name.endswith('Plot')]
print('Number of plot datasets  :', len(plot_refs))

by_type = collections.defaultdict(list)
for r in plot_refs:
    by_type[r.datasetType.name].append(r)

rows = []
for name, refs_ in sorted(by_type.items()):
    dt = refs_[0].datasetType
    rows.append(dict(
        datasetType=name,
        n=len(refs_),
        dimensions=', '.join(dt.dimensions.names),
        storageClass=dt.storageClass_name,
    ))

with pd.option_context('display.max_colwidth', None, 'display.width', 250):
    display(pd.DataFrame(rows))

### Choose a visit

In [ ]:
def get_visit(ref):
    '''Return the visit of a dataset reference, or None if the dataId has no visit.'''
    try:
        return ref.dataId['visit']
    except KeyError:
        return None


visits = sorted({v for v in (get_visit(r) for r in plot_refs) if v is not None})
print('Visits with plots :', visits)

# Change this index (or set VISIT_ID by hand) to look at another visit
VISIT_ID = visits[0] if visits else None
print('VISIT_ID =', VISIT_ID)

## Helpers : families of plots and display function

In [ ]:
# key : (title, regex on the dataset type name, short description)
# The descriptions are only a reading guide : the exact definition of each metric is in the
# lsst.analysis.tools code / docs, and the titles and labels of each figure are printed below it.
FAMILIES = {
    'astrom_repeat': (
        'Astrometric repeatability (histograms)',
        r'stellarAstrometricRepeatability[0-9]_HistPlot',
        'Distribution of the astrometric repeatability of stars (scatter of repeated position measurements). '
        'The digits 1, 2, 3 label the variants of the metric.'),
    'astrom_selfrepeat': (
        'Astrometric self-repeatability RA / Dec (histograms)',
        r'stellarAstrometricSelfRepeatability(RA|Dec)_HistPlot',
        'Same idea, separately for the RA and Dec coordinates.'),
    'astrom_resid_fp': (
        'Astrometric residuals RA / Dec (focal plane)',
        r'stellarAstrometricResiduals(RA|Dec)_FocalPlanePlot',
        'Astrometric residuals binned in focal-plane coordinates : position-dependent systematics (WCS, distortion).'),
    'astrom_resid_std_fp': (
        'Astrometric residual standard deviation RA / Dec (focal plane)',
        r'stellarAstrometricResidualStdDev(RA|Dec)_FocalPlanePlot',
        'Dispersion of the astrometric residuals across the focal plane.'),
    'photo_repeat': (
        'Photometric repeatability (histograms)',
        r'stellarPhotometricRepeatability(Calib)?_HistPlot',
        'Distribution of the photometric repeatability of stars. The Calib variant is the calibrated-flux version.'),
    'photo_resid_fp': (
        'Photometric residuals (focal plane)',
        r'stellarPhotometricResiduals(Calib)?_FocalPlanePlot',
        'Photometric residuals binned in focal-plane coordinates : detector- or position-dependent photometric systematics.'),
    'whole_sky': (
        'Whole-sky astrometric repeatability maps',
        r'whole_sky_.*WholeSkyPlot',
        'Sky maps of the astrometric repeatability metrics (AD1, AM2). With a COSMOS-only run only a small area is filled.'),
}


def family_types(key):
    '''Dataset type names belonging to a family.'''
    pattern = FAMILIES[key][1]
    return sorted(n for n in by_type if re.search(pattern, n))


def refs_of_type(name, visit=None):
    '''References of a dataset type for one visit (or all if visit is None / dataset has no visit).'''
    out = [r for r in by_type[name] if visit is None or get_visit(r) in (None, visit)]
    return sorted(out, key=lambda r: str(r.dataId))


def figure_titles(fig):
    '''Suptitle and axes titles of a matplotlib figure, to help reading the plot.'''
    titles = []
    sup = fig.get_suptitle()
    if sup:
        titles.append(sup)
    for ax in fig.axes:
        t = ax.get_title()
        if t:
            titles.append(t)
    return titles


def show_plot(ref, save=SAVE_FIGS):
    '''Retrieve one Plot dataset from the Butler (matplotlib Figure), display it and optionally save it.'''
    name = ref.datasetType.name
    visit = get_visit(ref)
    fig = butler.get(ref)

    display(Markdown(f'**{name}**  \n`{ref.dataId}`'))
    for t in figure_titles(fig):
        print('  title :', t)

    if save:
        base = os.path.join(FIGDIR, f'{name}_visit{visit}' if visit is not None else name)
        fig.savefig(base + '.png', dpi=100, bbox_inches='tight')
        fig.savefig(base + '.pdf', bbox_inches='tight')

    display(fig)
    plt.close(fig)


def show_family(key, visit=None):
    '''Display all the plots of a family for one visit.'''
    title, pattern, description = FAMILIES[key]
    display(Markdown(f'## {title}\n\n{description}'))
    names = family_types(key)
    if not names:
        print('No dataset type found for this family in the collection')
        return
    for name in names:
        for ref in refs_of_type(name, visit):
            show_plot(ref)

## Astrometric repeatability

In [ ]:
show_family('astrom_repeat', VISIT_ID)

In [ ]:
show_family('astrom_selfrepeat', VISIT_ID)

## Astrometric residuals in the focal plane

In [ ]:
show_family('astrom_resid_fp', VISIT_ID)

In [ ]:
show_family('astrom_resid_std_fp', VISIT_ID)

## Photometric repeatability and residuals

In [ ]:
show_family('photo_repeat', VISIT_ID)

In [ ]:
show_family('photo_resid_fp', VISIT_ID)

## Whole-sky maps

In [ ]:
show_family('whole_sky', VISIT_ID)

## Plot types not assigned to a family

In [ ]:
assigned = {n for key in FAMILIES for n in family_types(key)}
others = sorted(set(by_type) - assigned)
print('Dataset types without family :', others)

for name in others:
    for ref in refs_of_type(name, VISIT_ID):
        show_plot(ref)

## One plot type, all visits

In [ ]:
# Example : compare one plot type across all the visits of the collection
name = 'single_visit_star_association_stellarPhotometricRepeatability_HistPlot'

if name in by_type:
    for ref in refs_of_type(name):
        show_plot(ref)
else:
    print('Dataset type not found :', name)